# 05 – Ensemble Model & Final Comparison

This notebook:
1. Loads predictions from all base models
2. Trains a Ridge regression meta-learner on the validation set
3. Combines predictions to form the ensemble forecast
4. Computes final evaluation metrics and generates comparison plots

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
from pathlib import Path

from src.utils.data_loader import load_processed_data
from src.utils.metrics import calculate_metrics
from src.models.ensemble import EnsembleModel
from src.visualization.plotter import Plotter
from src.config import RESULTS_DIR, MODELS_SAVED_DIR

plotter = Plotter()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# ── Load data ────────────────────────────────────────────────────────────────
train = load_processed_data('train')
val   = load_processed_data('val')
test  = load_processed_data('test')

target_col = 'Close'
y_val  = val[target_col].values
y_test = test[target_col].values

In [4]:
# ── 1. Load Data & Train Meta-Learner ───────────────────────────────────────
import pandas as pd
from sklearn.linear_model import Ridge
from src.config import RESULTS_DIR
from src.utils.metrics import calculate_metrics
from src.visualization.plotter import Plotter

plotter = Plotter()
val_preds  = {}
test_preds = {}

# 1. 加载 2024 年各模型预测结果 (Phase 1)
df_val_base = pd.read_csv(RESULTS_DIR / 'baseline_val_preds.csv', index_col=0, parse_dates=True)
df_val_lstm = pd.read_csv(RESULTS_DIR / 'lstm_val_preds.csv', index_col=0, parse_dates=True)
df_val_lstm.columns = ['LSTM']

# 拼合成 Meta-Features 并对齐真实值
X_meta_train = df_val_base.join(df_val_lstm).dropna()
y_meta_train = val.loc[X_meta_train.index, 'Close']

# 2. 训练 Ridge Regression 作为 Ensemble
print("--- Training Ensemble Meta-Learner on 2024 Validation Data ---")
ensemble_model = Ridge(alpha=1.0)
ensemble_model.fit(X_meta_train, y_meta_train)

# 打印出极佳的报告素材：模型权重
print("\n🔍 Learned Ensemble Weights:")
weights = pd.Series(ensemble_model.coef_, index=X_meta_train.columns)
print(weights.sort_values(ascending=False))

# ── 2. Final Evaluation on 2025 Test Set ────────────────────────────────────
# 加载 2025 重新拟合后的预测 (Phase 2)
df_test_base = pd.read_csv(RESULTS_DIR / 'baseline_test_preds.csv', index_col=0, parse_dates=True)
df_test_lstm = pd.read_csv(RESULTS_DIR / 'lstm_test_preds.csv', index_col=0, parse_dates=True)
df_test_lstm.columns = ['LSTM']

X_meta_test = df_test_base.join(df_test_lstm).dropna()
y_true_2025 = test.loc[X_meta_test.index, 'Close']

# 生成最终预测
final_2025_preds = ensemble_model.predict(X_meta_test)
final_series = pd.Series(final_2025_preds, index=X_meta_test.index, name='Ensemble')

# 计算所有模型的终极得分，用于写进报告表格
all_pred_dict = {col: X_meta_test[col] for col in X_meta_test.columns}
all_pred_dict['Ensemble'] = final_series

all_metrics = {}
print("\n🏆 ULTIMATE 2025 TEST SET METRICS 🏆")
for name, preds in all_pred_dict.items():
    metrics = calculate_metrics(y_true_2025, preds)
    all_metrics[name] = metrics
    print(f"\n[{name}]")
    print(f"MSE:  {metrics['mse']:.2f} | RMSE: {metrics['rmse']:.2f}")
    print(f"MAE:  {metrics['mae']:.2f} | MAPE: {metrics['mape']:.2f}%")
    print(f"Dir Acc: {metrics['directional_accuracy'] * 100:.2f}%")

# ── 3. Final Visualisations ──────────────────────────────────────────────────
# 画出包含 Actual, ARIMA, Prophet, XGBoost, LSTM, Ensemble 的终极大图
plotter.plot_predictions_comparison(
    y_true=y_true_2025,
    predictions=all_pred_dict,
    dates=X_meta_test.index,
    title='2025 Final Showdown: All Base Models vs Ensemble',
    filename='ultimate_ensemble_comparison_2025.png'
)

# 生成一个对比柱状图，直观展示哪个模型 MAPE 最小
plotter.plot_metrics_comparison(all_metrics, filename='metrics_bar_chart_2025.png')
print("All final plots generated and saved to reports/figures/ !")

# ── 4. Save the Ultimate Metrics ─────────────────────────────────────────────
# 将字典转换为 DataFrame，转置 (T) 让模型作为行，指标作为列
final_metrics_df = pd.DataFrame(all_metrics).T

# 保存为全新的文件，避免和旧文件混淆
final_metrics_path = RESULTS_DIR / 'ultimate_2025_metrics.csv'
final_metrics_df.to_csv(final_metrics_path)

print(f"\n✅ Fresh metrics successfully saved to {final_metrics_path} !")
print(final_metrics_df)

# ── 5. Save the Ultimate Predictions Data (For Plotting) ───────────────────
# 把 2025 年真实的收盘价也加进去，作为基准对比列
final_preds_export = {'Actual_Close': y_true_2025}
final_preds_export.update(all_pred_dict)

# 将整个大字典转换为美观的 DataFrame
final_preds_df = pd.DataFrame(final_preds_export)

# 保存为 CSV 文件
final_preds_path = RESULTS_DIR / 'ultimate_2025_predictions.csv'
final_preds_df.to_csv(final_preds_path)

print(f"✅ All 2025 final predictions successfully saved to {final_preds_path} !")
print("\n预览前 5 天的预测数据：")
print(final_preds_df.head())

--- Training Ensemble Meta-Learner on 2024 Validation Data ---

🔍 Learned Ensemble Weights:
XGBoost    1.574780
LSTM       1.364909
Prophet    0.064801
ARIMA     -1.603494
dtype: float64

🏆 ULTIMATE 2025 TEST SET METRICS 🏆

[ARIMA]
MSE:  7133501.70 | RMSE: 2670.86
MAE:  2251.83 | MAPE: 9.66%
Dir Acc: 11.76%

[Prophet]
MSE:  3300785.88 | RMSE: 1816.81
MAE:  1356.44 | MAPE: 6.54%
Dir Acc: 59.24%

[XGBoost]
MSE:  14661710.72 | RMSE: 3829.06
MAE:  3274.12 | MAPE: 13.76%
Dir Acc: 47.06%

[LSTM]
MSE:  1576289.54 | RMSE: 1255.50
MAE:  1057.59 | MAPE: 4.49%
Dir Acc: 51.26%

[Ensemble]
MSE:  3177949.18 | RMSE: 1782.68
MAE:  1672.07 | MAPE: 7.48%
Dir Acc: 47.48%
All final plots generated and saved to reports/figures/ !

✅ Fresh metrics successfully saved to /Users/chenshuliu/Desktop/2_2_5152 ADA/2. Project/COMP5152ADA_Project_2/reports/results/ultimate_2025_metrics.csv !
                   mse         rmse          mae       mape  \
ARIMA     7.133502e+06  2670.861602  2251.833647   9.658365   


## Final Summary

| Metric | Best Model |
|--------|------------|
| RMSE   | See `reports/results/all_models_metrics.csv` |
| MAPE   | See `reports/results/all_models_metrics.csv` |
| Dir. Acc. | See `reports/results/all_models_metrics.csv` |

All results saved under `reports/results/`.